# Running associations between SPTSSB rs1450522 and PD clinical phenotypes
Since our local BCM CAND AD/PD dataset does not have a lot of power for subtle associations, neither has a lot of structured clinical data, I decided to run this analysis with GP2 (largest PD genetic cohort to date).

## 0. Setting up the environment

In [ ]:
## Useful packages and variables

import os                                       # interact with the environment
import sys                                      # interact with stderr
from firecloud import api as fapi               # interact with the workspace
from io import StringIO                         # work with file contents
import pandas as pd                             # work with tabular data
import numpy as np
import urllib.parse                             # build URLs to Google Cloud Console
from IPython.display import display, HTML       # display links to Google Cloud Console

# Enable IPython to display matplotlib graphs.
import matplotlib.pyplot as plt
%matplotlib inline
from cycler import cycler

# Get workspace attributes, so the notebook can access workspace data
# This was extracted from Terra, so will not necessarily work for Verily
# BILLING_PROJECT_ID = os.environ['GOOGLE_PROJECT']
# WORKSPACE_NAMESPACE = os.environ['WORKSPACE_NAMESPACE']
# WORKSPACE_NAME = os.environ['WORKSPACE_NAME']
# WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']

# You can still access Terra buckets through Verily, although it's recommended to simply use the mounted data
WORKSPACE_NAMESPACE = os.environ.get('WORKSPACE_NAMESPACE')
WORKSPACE_NAME = os.environ.get('WORKSPACE_NAME')
WORKSPACE_BUCKET = os.environ.get('WORKSPACE_BUCKET')
BILLING_PROJECT_ID = os.environ.get('GOOGLE_PROJECT')

WORKSPACE_ATTRIBUTES = fapi.get_workspace(WORKSPACE_NAMESPACE, WORKSPACE_NAME).json().get('workspace',{}).get('attributes',{})

In [ ]:
print("GOOGLE_PROJECT:", os.environ.get("GOOGLE_PROJECT"))
print("WORKSPACE_NAMESPACE:", os.environ.get("WORKSPACE_NAMESPACE"))
print("WORKSPACE_NAME:", os.environ.get("WORKSPACE_NAME"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))

In [ ]:
# Accessing files now is easier on Verily - no need to download, already in mounted data (can be mounted using Resources -> Data from catalog)

#!gcloud storage ls gs://gp2tier2/ --billing-project={BILLING_PROJECT_ID}
!ls -l ../gp2_tier2_eu_release12

In [ ]:
#!gcloud storage ls gs://gp2tier1/ --billing-project={BILLING_PROJECT_ID}

## 1. Prepare datasets

### First, look for clinical data

In [ ]:
#!gcloud storage ls gs://gp2tier2/release10/clinical_data/ --billing-project={BILLING_PROJECT_ID}
!ls -l ../gp2_tier2_eu_release12/clinical_data

In [ ]:
!head -30 ../gp2_tier2_eu_release12/clinical_data/r12_clinical_data_README.txt

In [ ]:
!head ../gp2_tier2_eu_release12/clinical_data/r12_extended_clinical_data.csv

In [ ]:
!ls -l

In [ ]:
#!mkdir clinical_data
# !gcloud storage cp \
#   gs://gp2tier2/release10/clinical_data/README_terra.txt ./clinical_data \
#   --billing-project=$GOOGLE_PROJECT

In [ ]:
# !gcloud storage cp \
#   gs://gp2tier2/release10/clinical_data/r10_extended_clinical_data_terra.csv ./clinical_data \
#   --billing-project=$GOOGLE_PROJECT

In [ ]:
%pip install gcsfs

In [ ]:
# Example of how to access a GP2 file, stored in a Terra bucket, without having to download it to my own storage

# First, install gcsfs (used by Pandas to access GCP storage)
import os
import pandas as pd

BILLING_PROJECT_ID = os.environ["GOOGLE_PROJECT"]

clinical_path = (
    "gs://gp2tier2/release10/clinical_data/"
    "r10_extended_clinical_data_terra.csv"
)

clinical = pd.read_csv(
    clinical_path,
    storage_options={
        "requester_pays": BILLING_PROJECT_ID
    }
)

clinical.head()

In [ ]:
# But what we actually need is release 12 Verily clinical data.
import os
import pandas as pd

clinical = pd.read_csv(
    "../gp2_tier2_eu_release12/clinical_data/r12_extended_clinical_data.csv"
)

#clinical.head()

# See all extended clinical data available
from IPython.display import display, HTML

table = pd.DataFrame({"column_name": clinical.columns})

display(HTML(f"""
<div style="
    max-height: 400px;
    overflow-y: auto;
    border: 1px solid #ddd;
">
    {table.to_html(index=False)}
</div>
"""))

Here are potential variables of interest for our analysis:
| Domain        | GP2 variable                       | Primary interpretation       |
| ------------- | ---------------------------------- | ---------------------------- |
| Motor         | `mds_updrs_part_iii_summary_score` | Objective motor severity     |
| Motor ADL     | `mds_updrs_part_ii_summary_score`  | Motor impact on daily living |
| Disease stage | `hoehn_and_yahr_stage`             | Global motor stage           |
| Non-motor     | `mds_updrs_part_i_summary_score`   | Broad non-motor burden       |
| Cognition     | `moca_total_score`                 | Cognitive performance        |
| Cognition     | `mmse_total_score`                 | Cognitive performance        |
| RBD           | `rbd_summary_score`                | REM sleep behavior symptoms  |
| Sleepiness    | `ess_total_score`                  | Daytime sleepiness           |
| Depression    | `gds15_total_score`                | Depressive symptoms          |
| Complications | `mds_updrs_part_iv_summary_score`  | Motor complications          |

### Then, look for genotype array data
(better than WGS because there are more PD cases with array data)

In [ ]:
#!gcloud storage ls gs://gp2tier2/release10/imputed_genotypes/EUR/ --billing-project={BILLING_PROJECT_ID}
!ls -l ../gp2_tier2_eu_release12/imputed_genotypes/EUR/

In [ ]:
#!mkdir -p rs1450522_GP2
#!mkdir -p rs1450522_GP2/genotypes

In [ ]:
# Install plink2
!wget https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_avx2_20250129.zip
!unzip plink2_linux_avx2_20250129.zip
!chmod 777 plink2

In [ ]:
# Extract the GRCh38 rs1450522 position
# Make sure plink is one folder before workspace and not inside mounted PIPELINES
!chmod 777 ../../plink2
!../../plink2 \
    --pfile "../gp2_tier2_eu_release12/imputed_genotypes/EUR/chr3_EUR_release12_vwb" \
    --chr 3 \
    --from-bp 161359842 \
    --to-bp 161359842 \
    --max-alleles 2 \
    --make-pgen \
    --out "./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12"

In [ ]:
# Inspect what was actually extracted
!head -30 ./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12.pvar

In [ ]:
# Explicitly tell PLINK to count G allele
!awk 'BEGIN{OFS="\t"} !/^#/ && $1==3 && $2==161359842 && $4=="A" && $5=="G" {print $3,"G"}' ./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12.pvar > ./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12.G.allele
!head ./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12.G.allele

!../../plink2 \
    --pfile "./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12" \
    --export A \
    --export-allele "./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12.G.allele" \
    --out "./rs1450522_GP2/rs1450522_imputed_genotypes_EUR_release12_G"